In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the image and convert to grayscale
image = cv2.imread("Eye image file name here")

if image is None:
    print("Error: Could not load 'eye.png'. Please check the file path.")
else:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gry = cv2.medianBlur(gray, 25)

    # 2. Detect circles using Hough Transform
    circles = cv2.HoughCircles(
        gry, cv2.HOUGH_GRADIENT, 1, 200, 
        param1=50, param2=50, minRadius=50, maxRadius=200
    )

    # Safety check: Ensure at least one circle was detected
    if circles is not None:
        circles = np.uint16(np.around(circles))
        x, y, r = circles[0][0]
        
        # Draw the detected iris circle on the original image
        cv2.circle(image, (x, y), r, (0, 255, 0), 3)

        # 3. Crop and isolate the iris region
        # Adding safety bounds so cropping doesn't go outside image dimensions
        h, w, _ = image.shape
        y_min, y_max = max(0, y - int(r * 0.7)), min(h, y + int(r * 0.7))
        x_min, x_max = max(0, x - int(r * 0.7)), min(w, x + int(r * 0.7))
        
        iris = image[y_min:y_max, x_min:x_max]
        iris_hsv = cv2.cvtColor(iris, cv2.COLOR_BGR2HSV)

        # 4. Color Masking and Analysis
        lower_limit = np.array([0, 40, 40])
        upper_limit = np.array([180, 255, 255])
        mask = cv2.inRange(iris_hsv, lower_limit, upper_limit)
        
        hue_channel = iris_hsv[:, :, 0]
        valid_hues = hue_channel[mask > 0]

        if len(valid_hues) > 0:
            average_hue = np.mean(valid_hues)
            
            # Classification based on average hue
            if average_hue < 20:
                color = "Brown"
            elif 20 <= average_hue < 85:
                color = "Green/Hazel"
            else:
                color = "Blue"
        else:
            average_hue = "N/A"
            color = "Unknown (No valid pixels masked)"

        # 5. Convert BGR to RGB for correct plotting colors in Matplotlib
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        iris_rgb = cv2.cvtColor(iris, cv2.COLOR_BGR2RGB)

        # 6. Display the Results
        fig, ax = plt.subplots(2, 1, figsize=(10, 15))
        
        ax[0].set_title(f"Average Hue Value: {average_hue}\nEye colour (possibly): {color}")
        ax[0].imshow(image_rgb)
        ax[0].axis('off')
        
        ax[1].imshow(iris_rgb)
        ax[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
    else:
        print("HoughCircles could not find any circles. Try tweaking param1, param2, or the radii.")